# Notebook 02 ? QLoRA Fine-Tuning (SFT)
**FinAlign | Mistral-7B-Instruct-v0.2 + 4-bit NF4 QLoRA on Colab A100**

Uses cleaned dataset from Notebook 01. Mirrors `src/train_qlora.py`.

## 0. Install

In [ ]:
!pip install -q transformers>=4.40.0 peft>=0.10.0 trl>=0.8.6 \
    bitsandbytes>=0.43.0 datasets>=2.18.0 accelerate>=0.28.0 \
    flash-attn==2.5.8 wandb scipy
print("All installed")

## 1. Config

In [ ]:
import os, json, torch

CFG = {
    "model_name"      : "mistralai/Mistral-7B-Instruct-v0.2",
    "max_seq_len"     : 2048,
    "lora_r"          : 64,
    "lora_alpha"      : 128,
    "lora_dropout"    : 0.05,
    "target_modules"  : ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    "per_device_batch": 4,
    "grad_accum"      : 4,
    "learning_rate"   : 2e-4,
    "num_epochs"      : 3,
    "warmup_ratio"    : 0.03,
    "lr_scheduler"    : "cosine",
    "weight_decay"    : 0.001,
    "max_grad_norm"   : 0.3,
    "train_file"      : "data/processed/train.jsonl",
    "val_file"        : "data/processed/val.jsonl",
    "output_dir"      : "checkpoints/sft_adapter",
    "wandb_project"   : "FinAlign-SFT",
    "wandb_run"       : "sft-mistral7b-r64-lr2e4",
    "seed"            : 42,
}

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print("Effective batch:", CFG["per_device_batch"] * CFG["grad_accum"])

## 2. W&B Init

In [ ]:
import wandb
wandb.login()
wandb.init(project=CFG["wandb_project"], name=CFG["wandb_run"],
           config=CFG, tags=["sft","qlora","mistral-7b","personal-finance"])
print(wandb.run.url)

## 3. Load & Format Dataset

In [ ]:
from datasets import load_dataset

TEMPLATE = "<s>[INST] {instruction} [/INST] {output}</s>"

def fmt(ex):
    instr = ex.get("instruction","").strip()
    ctx   = ex.get("input","").strip()
    out   = ex.get("output", ex.get("response","")).strip()
    prompt = f"{instr}\n\n{ctx}" if ctx else instr
    return {"text": TEMPLATE.format(instruction=prompt, output=out)}

ds = load_dataset("json",
    data_files={"train": CFG["train_file"], "validation": CFG["val_file"]},
    split=None)
ds = ds.map(fmt, remove_columns=ds["train"].column_names)
print(f"Train: {len(ds['train'])}  |  Val: {len(ds['validation'])}")
print("Sample text:", ds["train"][0]["text"][:400])

## 4. Load Quantised Model + LoRA

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from transformers import set_seed
set_seed(CFG["seed"])

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    CFG["model_name"], quantization_config=bnb,
    device_map="auto", torch_dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",
)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_cfg = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=CFG["lora_r"], lora_alpha=CFG["lora_alpha"],
    lora_dropout=CFG["lora_dropout"],
    target_modules=CFG["target_modules"],
    bias="none", inference_mode=False,
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()  # Expect ~1.16%

tokenizer = AutoTokenizer.from_pretrained(CFG["model_name"], padding_side="right")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

## 5. SFTTrainer Setup

In [ ]:
from transformers import EarlyStoppingCallback
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir=CFG["output_dir"], overwrite_output_dir=True,
    num_train_epochs=CFG["num_epochs"],
    per_device_train_batch_size=CFG["per_device_batch"],
    per_device_eval_batch_size=CFG["per_device_batch"],
    gradient_accumulation_steps=CFG["grad_accum"],
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_32bit",
    learning_rate=CFG["learning_rate"],
    weight_decay=CFG["weight_decay"],
    max_grad_norm=CFG["max_grad_norm"],
    lr_scheduler_type=CFG["lr_scheduler"],
    warmup_ratio=CFG["warmup_ratio"],
    bf16=True, tf32=True, fp16=False,
    eval_strategy="epoch", save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss", greater_is_better=False,
    save_total_limit=2,
    max_seq_length=CFG["max_seq_len"], packing=True,
    report_to="wandb", run_name=CFG["wandb_run"],
    seed=CFG["seed"], dataloader_num_workers=4,
    group_by_length=True, logging_steps=10, logging_first_step=True,
)

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=ds["train"], eval_dataset=ds["validation"],
    args=training_args,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3, early_stopping_threshold=1e-4)],
    dataset_text_field="text",
)
print("Trainer ready. Effective batch:", CFG["per_device_batch"] * CFG["grad_accum"])

## 6. Train

In [ ]:
trainer.train()

## 7. Save SFT Adapter

In [ ]:
import json
os.makedirs(CFG["output_dir"], exist_ok=True)
trainer.model.save_pretrained(CFG["output_dir"])
tokenizer.save_pretrained(CFG["output_dir"])
with open(os.path.join(CFG["output_dir"], "sft_meta.json"), "w") as f:
    json.dump(CFG, f, indent=2, default=str)
print(f"SFT adapter saved: {CFG['output_dir']}")
wandb.finish()

## 8. Quick Test

In [ ]:
from peft import PeftModel

tmodel = AutoModelForCausalLM.from_pretrained(
    CFG["model_name"], quantization_config=bnb, device_map="auto", torch_dtype=torch.bfloat16)
tmodel = PeftModel.from_pretrained(tmodel, CFG["output_dir"], is_trainable=False)
tmodel.eval()

def ask(q, max_tok=300, temp=0.7):
    prompt = f"<s>[INST] {q} [/INST]"
    inp = tokenizer(prompt, return_tensors="pt").to(tmodel.device)
    with torch.no_grad():
        out = tmodel.generate(**inp, max_new_tokens=max_tok,
            do_sample=True, temperature=temp, top_p=0.9,
            repetition_penalty=1.1, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)

print(ask("How should I start building an emergency fund?"))